In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
# from torchvision.models import 
import torchvision.models as models


import string
from tqdm.notebook import tqdm
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import multiprocessing as mp

In [2]:
csv_data_path = "/kaggle/input/khmer-dataset-final/new_dataset/labels/labels.csv"
image_data_path = "/kaggle/input/khmer-dataset-final/new_dataset/images/Battambang/"

In [3]:
import pandas as pd


df = pd.read_csv(csv_data_path)
df


,image,label
0,Khmer_txt_1.png,ហេតុដូចម្តេចទើបមានការបង្កើតតាក់ស៊ីក្រហមឬTaxiRo...
1,Khmer_txt_2.png,និយាយទៅខ្ញុំបានសង្កេតពីការវិវត្តរបស់ប្រទេសកម្ព...
2,Khmer_txt_3.png,ខ្ញុំក៏សម្រេចចិត្តដំណើរការក្រុមហ៊ុនផ្តល់សេវាកម...
3,Khmer_txt_4.png,ហេតុអ្វីបានជាតាក់ស៊ីក្រហមត្រូវតែមាននៅកម្ពុជា?
4,Khmer_txt_5.png,តើចំណុចណាដែលធ្វើឲ្យតាក់ស៊ីក្រហមលេចធ្លោជាងក្រុម...
...,...,...
5244,Khmer_txt_5245.png,ការវេចខ្ចប់40HQ240HQ240HQ2
5245,Khmer_txt_5246.png,សេវាSathapana
5246,Khmer_txt_5247.png,សីលាបើមិនធ្វើមានតែអត់។
5247,Khmer_txt_5248.png,សំនួរការទាញយកhighdefinitionenvironmentv21.zip


In [4]:
df = df.dropna()

def clean_irrelevant(result):    
    result = result.replace('\u200b','')
    result = result.replace('/u200b','')

    # result = result.replace('quot ','')

    return result
df["label"] = df["label"].apply(clean_irrelevant)
# df = df[df['label'].apply(lambda x: len(x) <= 40)]
# df= df.reset_index(drop=True)

df

,image,label
0,Khmer_txt_1.png,ហេតុដូចម្តេចទើបមានការបង្កើតតាក់ស៊ីក្រហមឬTaxiRo...
1,Khmer_txt_2.png,និយាយទៅខ្ញុំបានសង្កេតពីការវិវត្តរបស់ប្រទេសកម្ព...
2,Khmer_txt_3.png,ខ្ញុំក៏សម្រេចចិត្តដំណើរការក្រុមហ៊ុនផ្តល់សេវាកម...
3,Khmer_txt_4.png,ហេតុអ្វីបានជាតាក់ស៊ីក្រហមត្រូវតែមាននៅកម្ពុជា?
4,Khmer_txt_5.png,តើចំណុចណាដែលធ្វើឲ្យតាក់ស៊ីក្រហមលេចធ្លោជាងក្រុម...
...,...,...
5244,Khmer_txt_5245.png,ការវេចខ្ចប់40HQ240HQ240HQ2
5245,Khmer_txt_5246.png,សេវាSathapana
5246,Khmer_txt_5247.png,សីលាបើមិនធ្វើមានតែអត់។
5247,Khmer_txt_5248.png,សំនួរការទាញយកhighdefinitionenvironmentv21.zip


In [5]:
def check_missing_values(df):
    """
    This function checks if there are any missing values in any row of the DataFrame.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to check for missing values.
    
    Returns:
    bool: True if there are missing values in any row, False otherwise.
    """
    return df.isnull().any(axis=1).any()

if check_missing_values(df):
    print("There are missing values in the DataFrame.")
else:
    print("There are no missing values in the DataFrame.")

There are no missing values in the DataFrame.


In [6]:
datapath_all = df["image"].tolist()
label_all = df["label"].tolist()
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# print((train_df["Label"]))
def check_missing_values(df):
    """
    This function checks if there are any missing values in any row of the DataFrame.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to check for missing values.
    
    Returns:
    bool: True if there are missing values in any row, False otherwise.
    """
    return df.isnull().any(axis=1).any()
datapath_train = train_df["image"].tolist()
datapath_test = test_df["image"].tolist()

label_train = train_df["label"].tolist()
label_test = test_df["label"].tolist()
label_train[:5]

['ក្រសួងនេះបានឲ្យដឹងនៅក្នុងសេចក្ដីប្រកាសព័ត៌មានមួយថា៖',
 'ទំហំក្រឡា33x36x90mm',
 'វត្ថុត្រាប់តាមComment',
 'ឈ្មោះម៉ាកhanglongzhongtian',
 'ខ្ញុំទើបតែបានការងារថ្មីនៅទីក្រុងដូច្នេះខ្ញុំដើរទិញសំលៀកបំពាក់។']

In [7]:
print(len(label_train))
print(np.unique([len(image_fn.split(" ")[0]) for image_fn in label_train]))

4199
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72
 73 74 75 76 77 78 79 80 81 82 83 84]


In [8]:
cpu_count = mp.cpu_count()
print(cpu_count)

4


In [9]:
# image_ns = [image_fn.split(".")[0] for image_fn in label_all]
image_ns = [image_fn.split(" ")[0] for image_fn in label_all]


# image_ns = [image_fn[0] for image_fn in label_all]


image_ns = "".join(image_ns)
# letters = sorted(list(set(list(image_ns))))
# print(len(letters))
# print(letters)

In [10]:
import json
import os

# Path to your vocabulary JSON file
vocab_json_path = '/kaggle/input/dictionary/vocab.json'

# Check if the file exists before loading
with open(vocab_json_path, 'r') as f:
    vocab_dict = json.load(f)

    num2char = vocab_dict['idx2char']
    char2num = vocab_dict['char2idx']

    print("Vocabulary loaded from vocab.json")
    print("num2char:", num2char)
    print("char2num:", char2num)



Vocabulary loaded from vocab.json
num2char: {'0': '!', '1': '$', '2': '%', '3': '&', '4': "'", '5': '(', '6': ')', '7': '+', '8': ',', '9': '-', '10': '.', '11': '/', '12': '0', '13': '1', '14': '2', '15': '3', '16': '4', '17': '5', '18': '6', '19': '7', '20': '8', '21': '9', '22': ':', '23': '?', '24': 'A', '25': 'B', '26': 'C', '27': 'D', '28': 'E', '29': 'F', '30': 'G', '31': 'H', '32': 'I', '33': 'J', '34': 'K', '35': 'L', '36': 'M', '37': 'N', '38': 'O', '39': 'P', '40': 'Q', '41': 'R', '42': 'S', '43': 'T', '44': 'U', '45': 'V', '46': 'W', '47': 'X', '48': 'Y', '49': 'Z', '50': '[', '51': ']', '52': 'a', '53': 'b', '54': 'c', '55': 'd', '56': 'e', '57': 'f', '58': 'g', '59': 'h', '60': 'i', '61': 'j', '62': 'k', '63': 'l', '64': 'm', '65': 'n', '66': 'o', '67': 'p', '68': 'q', '69': 'r', '70': 's', '71': 't', '72': 'u', '73': 'v', '74': 'w', '75': 'x', '76': 'y', '77': 'z', '78': '«', '79': '»', '80': 'ç', '81': 'è', '82': 'é', '83': 'ក', '84': 'ខ', '85': 'គ', '86': 'ឃ', '87': 'ង

In [11]:
batch_size = 16

In [12]:
class CAPTCHADataset(Dataset):
    
    def __init__(self, data_dir, image_fns):
        self.data_dir = data_dir
        self.image_fns = image_fns
        
    def __len__(self):
        return len(self.image_fns)
    
    def __getitem__(self, index):
        image_fn = self.image_fns[index]
        image_fp = os.path.join(self.data_dir, image_fn)
        #image_fp = image_fp + ".png"
        image = Image.open(image_fp).convert('RGB')
        image = self.transform(image)
#         text = image_fn.split(".")[0]
        text = df.loc[df['image']==image_fn,"label"].iloc[0]
#         print(text)
        
        return image, text
    
    def transform(self, image):
        
        transform_ops = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
        ])
        return transform_ops(image)

In [13]:
trainset = CAPTCHADataset(image_data_path, datapath_train) 
testset = CAPTCHADataset(image_data_path, datapath_test)
train_loader = DataLoader(trainset, batch_size=batch_size, num_workers=cpu_count, shuffle=True)
test_loader = DataLoader(testset, batch_size=batch_size, num_workers=cpu_count, shuffle=False)
print(len(train_loader), len(test_loader))

263 66


In [14]:
image_batch, text_batch = next(iter(train_loader))
print(image_batch.size(), text_batch)

/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


torch.Size([16, 3, 100, 1400]) ('ស្ងាត់ៗJames', 'សន្និបាតស្ដីអំពីទីក្រុងរបស់យើងទាំងអស់គ្នាឆ្នាំ២០១៨', 'បើស្រុកនេះទិញមិនស្រួលមិនប៉ះអារបស់ធ្វើអោយសវាទៅជាអាធ្វើអោយខ្មៅទៅវិញទៀតណឹងណា', 'ពីភស្តុតាងទៅជាអត្ថបទ', 'លោកBrianSinclair-Thompsonអនុប្រធានJFCCTបានថ្លែងថា៖', 'ច្បាប់', 'ប្រាក់អឺរ៉ូស្ទើរតែមានតម្លៃទ្រឹងដោយជួញដូរក្នុងតម្លៃ១,៣៥៣៩ដុល្លារក្នុងមួយអឺរ៉ូ៕', 'អ្នកមិនអាចពិភាក្សាពីយុទ្ធសាស្ដ្រនិងគោលនយោបាយនៅតាមដងផ្លូវទេវាមិនអាចទៅរួចទេ»។', 'អង្គភាពគណ', 'មជ្ឈមណ្ឌលព', 'យើងត្រូវភ្ជាប់ឡើងវិញជាមួយទំនាក់ទំនងរបស់1WorldMarch។', 'បាទសូមអញ្ជើញ។', 'មានបោះដុំរាយយកកាន់តែច្រើនតំលៃកាន់តែធូ', 'លោកមិនទាន់បានសម្រេចចិត្តទេដោយបញ្ជាក់ថា៖', 'ប្រវែង505ម៉ាយ813គីឡូម៉ែត្រ', 'ការត្រួតពិនិត្យគោលដៅអភិវឌ្ឍប្រកបដោយចីរភាពទី២នៅក្នុងប្រទេសកម្ពុជា')


In [15]:
num_chars = len(char2num)
print(num_chars)
rnn_hidden_size = 512
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

172
cpu


In [16]:
print(len(char2num))
print(char2num)
print(len(num2char))
print(num2char)


172
{'!': 0, '$': 1, '%': 2, '&': 3, "'": 4, '(': 5, ')': 6, '+': 7, ',': 8, '-': 9, '.': 10, '/': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, ':': 22, '?': 23, 'A': 24, 'B': 25, 'C': 26, 'D': 27, 'E': 28, 'F': 29, 'G': 30, 'H': 31, 'I': 32, 'J': 33, 'K': 34, 'L': 35, 'M': 36, 'N': 37, 'O': 38, 'P': 39, 'Q': 40, 'R': 41, 'S': 42, 'T': 43, 'U': 44, 'V': 45, 'W': 46, 'X': 47, 'Y': 48, 'Z': 49, '[': 50, ']': 51, 'a': 52, 'b': 53, 'c': 54, 'd': 55, 'e': 56, 'f': 57, 'g': 58, 'h': 59, 'i': 60, 'j': 61, 'k': 62, 'l': 63, 'm': 64, 'n': 65, 'o': 66, 'p': 67, 'q': 68, 'r': 69, 's': 70, 't': 71, 'u': 72, 'v': 73, 'w': 74, 'x': 75, 'y': 76, 'z': 77, '«': 78, '»': 79, 'ç': 80, 'è': 81, 'é': 82, 'ក': 83, 'ខ': 84, 'គ': 85, 'ឃ': 86, 'ង': 87, 'ច': 88, 'ឆ': 89, 'ជ': 90, 'ឈ': 91, 'ញ': 92, 'ដ': 93, 'ឋ': 94, 'ឌ': 95, 'ឍ': 96, 'ណ': 97, 'ត': 98, 'ថ': 99, 'ទ': 100, 'ធ': 101, 'ន': 102, 'ប': 103, 'ផ': 104, 'ព': 105, 'ភ': 106, 'ម': 107, 'យ': 108, 'រ': 109, 'ល': 

In [17]:
class OCR_CNN_GRU(nn.Module):
    def __init__(self, num_chars=num_chars, rnn_hidden_size=512, dropout=0.1):
        super(OCR_CNN_GRU, self).__init__()
        self.num_chars = num_chars
        self.rnn_hidden_size = rnn_hidden_size
        self.dropout = dropout
        
        # CNN layers
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=2),
            
            nn.Conv2d(64, 128, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=2),
        )

        self.linear1 = nn.Linear(3200, rnn_hidden_size)


        
        # GRU layers
        self.gru = nn.GRU(
            input_size=rnn_hidden_size, 
            hidden_size=rnn_hidden_size, 
            num_layers=2, 
            bidirectional=True, 
            batch_first=True
        )
        
        # Output layer
        self.linear2 = nn.Linear(rnn_hidden_size * 2, num_chars)
        
    def forward(self, x):
        x = self.cnn(x)
        
        # Reshape from [batch_size, channels, height, width] to [batch_size, width, channels * height]
        batch_size, channels, height, width = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(batch_size, width, -1)
        
        # Linear layer to reduce dimension
        x = self.linear1(x)
        
        # GRU layers
        x, _ = self.gru(x)
        
        # Output layer
        x = self.linear2(x)
        
        # Transpose to get the shape [sequence_length, batch_size, num_classes]
        x = x.transpose(0, 1)
        
        return x

def weights_init(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            m.bias.data.fill_(0.01)
    elif isinstance(m, nn.BatchNorm2d):
        m.weight.data.fill_(1.0)
        m.bias.data.fill_(0.01)




In [18]:
criterion = nn.CTCLoss(blank=0)
def encode_text_batch(text_batch):
    
    text_batch_targets_lens = [len(text) for text in text_batch]
    text_batch_targets_lens = torch.IntTensor(text_batch_targets_lens)
    
    text_batch_concat = "".join(text_batch)
    text_batch_targets = [char2num[c] for c in text_batch_concat]
    text_batch_targets = torch.IntTensor(text_batch_targets)
    
    return text_batch_targets, text_batch_targets_lens

In [19]:
def compute_loss(text_batch, text_batch_logits):
    """
    text_batch: list of strings of length equal to batch size
    text_batch_logits: Tensor of size([T, batch_size, num_classes])
    """
    text_batch_logps = F.log_softmax(text_batch_logits, 2) # [T, batch_size, num_classes]  
    text_batch_logps_lens = torch.full(size=(text_batch_logps.size(1),), 
                                       fill_value=text_batch_logps.size(0), 
                                       dtype=torch.int32).to(device) # [batch_size] 
    #print(text_batch_logps.shape)
    #print(text_batch_logps_lens) 
    text_batch_targets, text_batch_targets_lens = encode_text_batch(text_batch)
    #print(text_batch_targets)
    #print(text_batch_targets_lens)
    loss = criterion(text_batch_logps, text_batch_targets, text_batch_logps_lens, text_batch_targets_lens)

    return loss

# compute_loss(text_batch, text_batch_logits)

In [20]:
# Define the device (CPU)
device = torch.device('cpu')

# Load the model with map_location specified
model = OCR_CNN_GRU()
model.load_state_dict(torch.load('/kaggle/input/checkpoint/checkpoint.pth', map_location=device))


<All keys matched successfully>

In [21]:
def decode_predictions(text_batch_logits):

    text_batch_tokens = F.softmax(text_batch_logits, 2).argmax(2) # [T, batch_size]
    text_batch_tokens = text_batch_tokens.numpy().T # [batch_size, T]

    text_batch_tokens_new = []
    for text_tokens in text_batch_tokens:
        text = [num2char[str(idx)] for idx in text_tokens]
        text = "".join(text)
        text_batch_tokens_new.append(text)

    return text_batch_tokens_new

In [22]:
df

,image,label
0,Khmer_txt_1.png,ហេតុដូចម្តេចទើបមានការបង្កើតតាក់ស៊ីក្រហមឬTaxiRo...
1,Khmer_txt_2.png,និយាយទៅខ្ញុំបានសង្កេតពីការវិវត្តរបស់ប្រទេសកម្ព...
2,Khmer_txt_3.png,ខ្ញុំក៏សម្រេចចិត្តដំណើរការក្រុមហ៊ុនផ្តល់សេវាកម...
3,Khmer_txt_4.png,ហេតុអ្វីបានជាតាក់ស៊ីក្រហមត្រូវតែមាននៅកម្ពុជា?
4,Khmer_txt_5.png,តើចំណុចណាដែលធ្វើឲ្យតាក់ស៊ីក្រហមលេចធ្លោជាងក្រុម...
...,...,...
5244,Khmer_txt_5245.png,ការវេចខ្ចប់40HQ240HQ240HQ2
5245,Khmer_txt_5246.png,សេវាSathapana
5246,Khmer_txt_5247.png,សីលាបើមិនធ្វើមានតែអត់។
5247,Khmer_txt_5248.png,សំនួរការទាញយកhighdefinitionenvironmentv21.zip


In [23]:
import os
import torch
import pandas as pd
from PIL import Image
import torch.nn.functional as F
from torchvision import transforms
import json

# Define the transformation operations
transform_ops = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

# Function to remove duplicates in the predicted text
def remove_duplicates(text):
    if len(text) > 1:
        letters = [text[0]] + [letter for idx, letter in enumerate(text[1:], start=1) if text[idx] != text[idx-1]]
    elif len(text) == 1:
        letters = [text[0]]
    else:
        return ""
    return "".join(letters)

# Function to correct the predicted word
def correct_prediction(word):
    parts = word.split("-")
    parts = [remove_duplicates(part) for part in parts]
    corrected_word = "".join(parts)
    return corrected_word

# Function to decode predictions from logits
def decode_predictions(text_batch_logits):
    text_batch_tokens = F.softmax(text_batch_logits, 2).argmax(2)  # [T, batch_size]
    text_batch_tokens = text_batch_tokens.numpy().T  # [batch_size, T]

    text_batch_tokens_new = []
    for text_tokens in text_batch_tokens:
        text = [num2char[str(idx)] for idx in text_tokens]
        text = "".join(text)
        text_batch_tokens_new.append(text)

    return text_batch_tokens_new

# Inference function
def infer_image(image_path, model, device):
    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")
    image = transform_ops(image).unsqueeze(0).to(device)  # Add batch dimension

    # Perform inference
    model.eval()
    with torch.no_grad():
        text_logits = model(image)  # [T, batch_size, num_classes]
        text_logits = text_logits.cpu()

    # Decode predictions
    text_pred = decode_predictions(text_logits)[0]

    # Apply post-processing corrections
    text_pred_corrected = correct_prediction(text_pred)

    return text_pred_corrected

# Directory containing images
image_directory = '/kaggle/input/test-dataset-bms/test_dataset_siemreap/images/Siemreap'

# Model initialization and loading weights
# Ensure that you have defined and initialized your model class correctly
# Example:
# crnn = OCR_CNN_GRU(num_chars=len(idx2char), rnn_hidden_size=512, dropout=0.5)  # Replace with your model initialization
# crnn.load_state_dict(torch.load('path/to/your/model.pth', map_location=torch.device('cpu')))  # Load the trained model weights
# crnn.to(device)

# Ensure to use the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Iterate over all images in the directory and perform inference
results = []
for filename in os.listdir(image_directory):
    if filename.endswith('.png') or filename.endswith('.jpg'):
        image_path = os.path.join(image_directory, filename)
        predicted_text = infer_image(image_path, model, device)
        results.append({'filename': filename, 'predicted_text': predicted_text})

# Convert the results to a DataFrame and optionally save to a CSV file
dfs = pd.DataFrame(results)

# Display the DataFrame
dfs


              filename                                     predicted_text
0    Khmer_txt_200.png  !គុណ!ប្រ!យ!ោ!ជ!ន៍នការ!ព្យាបា!ល!សម្មស្ស!ដោ!យ!កា...
1    Khmer_txt_297.png  !មិន!ត្រឹម!មាន!ស!ម្ត្ភាពក្នុងវិស័យ!សិល្បៈ!នោះ!...
2    Khmer_txt_218.png  !អត្ថ!ប!ទ!៖!F!a!celne!C!a!m!b!o!dia!S!h!arethi...
3    Khmer_txt_253.png  !លើស!ពី!នេះការពិភាក្សាគ្នារវាងនិស្សិត!នៃសាក!ល!...
4    Khmer_txt_312.png  !យោង!តាម!ប្រ!ភ!ព!ព័ត៌!មាន!ច្បាស់ការ!ថាជ!ន!រង!គ...
..                 ...                                                ...
96   Khmer_txt_334.png  !រុរក្ត!F!alico!n!9របស់ក្រុមហ៊ុន!ឯ!ក!ជ!នមួយ!នេ...
97   Khmer_txt_206.png  !ការព្យាបាល!ពិសេ!ស!នេះ!ជាញឹកញ!ាប់ត្រូវ!បានប្រើ...
98   Khmer_txt_322.png  !ក្រុម!បាតុក!របន្តវ៉ានៅអាកា!ស!យ!ានដ្ឋាន!ហុង!កុ...
99   Khmer_txt_263.png  !គរម្លឹក!ផ!ងដែរថាបច្ចុប្បន្នC!U!S!មាន!ទីតាំងគឺ...
100  Khmer_txt_308.png  !ឯ!ក!ខ្ត្តមរដ្មន្ីមើល!យើញ!ថាគ!ម្រោង!នេះនឹងធ្វើ...

[101 rows x 2 columns]


In [24]:
def remove_sign(text):
    text = text.replace("!","")
    return text
dfs["predicted_text"] = dfs["predicted_text"].apply(remove_sign)
dfs

,filename,predicted_text
0,Khmer_txt_200.png,គុណប្រយោជន៍នការព្យាបាលសម្មស្សដោយការចាក់បូថុក(B...
1,Khmer_txt_297.png,មិនត្រឹមមានសម្ត្ភាពក្នុងវិស័យសិល្បៈនោះទេសព្វថ្...
2,Khmer_txt_218.png,អត្ថបទ៖FacelneCambodiaSharethisarticlewilh.Sha...
3,Khmer_txt_253.png,លើសពីនេះការពិភាក្សាគ្នារវាងនិស្សិតនៃសាកលវិទ្យា...
4,Khmer_txt_312.png,យោងតាមប្រភពព័ត៌មានច្បាស់ការថាជនរងគ្រោះឈ្មោះជឹម...
...,...,...
96,Khmer_txt_334.png,រុរក្តFalicon9របស់ក្រុមហ៊ុនឯកជនមួយនេះបានហោះគេច...
97,Khmer_txt_206.png,ការព្យាបាលពិសេសនេះជាញឹកញាប់ត្រូវបានប្រើប្រាស់រ...
98,Khmer_txt_322.png,ក្រុមបាតុករបន្តវ៉ានៅអាកាសយានដ្ឋានហុងកុងប្រឆាំង...
99,Khmer_txt_263.png,គរម្លឹកផងដែរថាបច្ចុប្បន្នCUSមានទីតាំងគឺ២ទីតាំង...


In [25]:
ground_truth_path = '/kaggle/input/test-dataset-bms/test_dataset_siemreap/labels/labels.csv'  # Update with the actual path to your ground truth CSV
ground_truth_df = pd.read_csv(ground_truth_path)
ground_truth_df

,image,label
0,Khmer_txt_200.png,គុណប្រយោជន៍នៃការព្យាបាលសម្ផស្សដោយការចាក់បូថុក(...
1,Khmer_txt_201.png,បុរសស្រ្ដីគ្រប់រូបគួរតែដោះស្រាយបញ្ហាស្បែកជ្រីវ...
2,Khmer_txt_202.png,ទោះជាយ៉ាងណាក៏នៅមានវិធីសាស្រ្ដដ៏ប្រសើរមួយដើម្បី...
3,Khmer_txt_203.png,តើអ្វីទៅជាការព្យាបាលសម្ផស្សដោយBotox?Botoxគឺជាភ...
4,Khmer_txt_204.png,Botoxមានផ្ទុកនូវទម្រង់Toxinដែលត្រូវបានលាយផ្សំជ...
...,...,...
96,Khmer_txt_332.png,អាមេរិកៈក្រុមហ៊ុនអវកាសអាមេរិកSpaceXបានបាញ់រ៉ុក...
97,Khmer_txt_333.png,បច្ចុប្បន្នយានមួយនេះកំពុងធ្វើដំណើរតាមផ្លូវឆ្ពោ...
98,Khmer_txt_334.png,រ៉ុក្កែតFalcon9របស់ក្រុមហ៊ុនឯកជនមួយនេះបានហោះគេ...
99,Khmer_txt_335.png,ការដឹកជញ្ជូនលើកនេះដែរនឹងចូលចតក្នុងស្ថានីយ៍អវកា...


In [26]:
# Sort the ground truth DataFrame by 'image' column to match the order of filenames in dfs
ground_truth_df = ground_truth_df.sort_values(by='image').reset_index(drop=True)

# Merge predicted results with ground truth based on filenames
merged_df = pd.merge(dfs, ground_truth_df, left_on='filename', right_on='image')

# Function to calculate Character Error Rate (CER)
def calculate_cer(predicted_text, actual_text):
    predicted_text = predicted_text.strip()
    actual_text = actual_text.strip()
    errors = sum(1 for a, b in zip(predicted_text, actual_text) if a != b)
    errors += abs(len(predicted_text) - len(actual_text))
    return errors / max(len(actual_text), 1)  # Avoid division by zero

# Calculate CER for each row
merged_df['cer'] = merged_df.apply(lambda row: calculate_cer(row['predicted_text'], row['label']), axis=1)

# Calculate the average CER
average_cer = merged_df['cer'].mean()
print(f'Average CER: {average_cer}')

# Display the merged DataFrame with CER
merged_df[['filename', 'predicted_text', 'label', 'cer']]


Average CER: 0.5553563846535298


,filename,predicted_text,label,cer
0,Khmer_txt_200.png,គុណប្រយោជន៍នការព្យាបាលសម្មស្សដោយការចាក់បូថុក(B...,គុណប្រយោជន៍នៃការព្យាបាលសម្ផស្សដោយការចាក់បូថុក(...,0.655738
1,Khmer_txt_297.png,មិនត្រឹមមានសម្ត្ភាពក្នុងវិស័យសិល្បៈនោះទេសព្វថ្...,មិនត្រឹមមានសមត្ថភាពក្នុងវិស័យសិល្បៈនោះទេសព្វថ្...,0.034188
2,Khmer_txt_218.png,អត្ថបទ៖FacelneCambodiaSharethisarticlewilh.Sha...,អត្ថបទ៖FacelineCambodiaSharethisarticlewith:Sh...,0.889764
3,Khmer_txt_253.png,លើសពីនេះការពិភាក្សាគ្នារវាងនិស្សិតនៃសាកលវិទ្យា...,លើសពីនេះការពិភាក្សាគ្នារវាងនិស្សិតនៃសាកលវិទ្យា...,0.336735
4,Khmer_txt_312.png,យោងតាមប្រភពព័ត៌មានច្បាស់ការថាជនរងគ្រោះឈ្មោះជឹម...,យោងតាមប្រភពព័ត៌មានច្បាស់ការថាជនរងគ្រោះឈ្មោះជឹម...,0.443662
...,...,...,...,...
96,Khmer_txt_334.png,រុរក្តFalicon9របស់ក្រុមហ៊ុនឯកជនមួយនេះបានហោះគេច...,រ៉ុក្កែតFalcon9របស់ក្រុមហ៊ុនឯកជនមួយនេះបានហោះគេ...,0.973118
97,Khmer_txt_206.png,ការព្យាបាលពិសេសនេះជាញឹកញាប់ត្រូវបានប្រើប្រាស់រ...,ការព្យាបាលពិសេសនេះជាញឹកញាប់ត្រូវបានប្រើប្រាស់រ...,0.167598
98,Khmer_txt_322.png,ក្រុមបាតុករបន្តវ៉ានៅអាកាសយានដ្ឋានហុងកុងប្រឆាំង...,ក្រុមបាតុករបន្តតវ៉ានៅអាកាសយានដ្ឋានហុងកុងប្រឆាំ...,0.797297
99,Khmer_txt_263.png,គរម្លឹកផងដែរថាបច្ចុប្បន្នCUSមានទីតាំងគឺ២ទីតាំង...,គួររម្លឹកផងដែរថាបច្ចុប្បន្នCUSមាន៨ទីតាំងគឺ២ទីត...,0.962963


In [ ]:
import torch

# Load the checkpoint and map tensors to CPU
# checkpoint_path = '/kaggle/input/checkpoint/checkpoint.pth'
# checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))

# import torch

# Load the checkpoint
checkpoint_path = '/kaggle/input/checkpoint/checkpoint.pth'
checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))

# Print the keys of the checkpoint dictionary
print(checkpoint.keys())
# Print the best validation loss value
print("Best Validation Loss:", checkpoint['best_val_loss'])

